# Class 1 — Admission control

Every request passes a series of gates before your service does any expensive work. This notebook walks each gate and shows the status code it returns. **Predict each result before you run it.**

## How to run this class

1. In a terminal, build the database once:  `python create_db.py`
2. In the same terminal, start the server:  `uvicorn app:app --reload`
3. Run the cells below, top to bottom. Cell 4 checks the server is up.

Between scenes the cells call `/_reset`, which puts the demo back to a clean state.

## The gates

| gate | status | question |
|---|---|---|
| authentication | 401 | who are you? |
| rate limit | 429 | how fast? |
| validation | 422 | is the body well-formed? |
| authorization | 403 | are you allowed? |
| quota / credits | 402 | have you paid? |


## The two keys — don't mix them up

* **caller key** — the customer's app sends it to **your** service (`alice-123`).
* **provider key** — **your** service sends it to OpenRouter/OpenAI.

The caller key proves who is calling you. The provider key proves you may call them.

In [ ]:
import requests

BASE = "http://127.0.0.1:8000"

# Start the server first, in a terminal:  uvicorn app:app --reload
r = requests.get(f"{BASE}/health")
print(r.status_code, r.json())


In [ ]:
def call(method, path, key=None, body=None):
    headers = {"x-api-key": key} if key else {}
    return requests.request(method, f"{BASE}{path}", headers=headers, json=body)


## Demo — use the app, then look inside

Before we explain each gate, use the app the way a client would. Then open `class1.db` in your SQLite extension and watch `users.credits` and the `usage` table change.

In [ ]:
requests.post(f"{BASE}/_reset")   # start from a clean state

for i, tokens in enumerate([150, 200], 1):
    r = call("POST", "/v1/analyze", key="alice-123", body={"question": f"call {i}", "tokens": tokens})
    print(f"call {i}: {r.status_code}  charged {r.json()['tokens_charged']} tokens")


In [ ]:
import sqlite3
conn = sqlite3.connect("class1.db")   # run this notebook from the 01_admission_control folder
conn.row_factory = sqlite3.Row

print("users — alice's credits dropped from 1000:")
for row in conn.execute("SELECT user_id, name, monthly_tokens, credits FROM users"):
    print("   ", dict(row))

print("\nusage — two new rows, one per call:")
for row in conn.execute("SELECT user_id, tokens, ts FROM usage"):
    print("   ", dict(row))
conn.close()


## 1. Authentication — who are you?

No key, a wrong key, and the right key:

In [ ]:
requests.post(f"{BASE}/_reset")   # clean slate after the demo
r = call("POST", "/v1/analyze", body={"question": "hi"})
print("no key:   ", r.status_code, r.json()["detail"])

r = call("POST", "/v1/analyze", key="nope", body={"question": "hi"})
print("wrong key:", r.status_code, r.json()["detail"])

r = call("POST", "/v1/analyze", key="alice-123", body={"question": "hi", "tokens": 100})
print("alice:    ", r.status_code, r.json())


## Under the hood: why a hash, not the key

The app accepted `alice-123` — but it never stores `alice-123`. It stores a **fingerprint** (a SHA-256 hash). On every request it fingerprints what you sent and compares. The database can leak and no working key comes out of it.

In [ ]:
import sqlite3, hashlib

conn = sqlite3.connect("class1.db")   # run this notebook from the 01_admission_control folder
stored = dict(conn.execute("SELECT user_id, key_hash FROM users").fetchall())
conn.close()

def fingerprint(s):
    return hashlib.sha256(s.encode()).hexdigest()

print("stored for alice:", stored["alice"])
print("sha256('alice-123') == stored ?", fingerprint("alice-123") == stored["alice"])
print("sha256('guess')     == stored ?", fingerprint("guess") == stored["alice"])


## 2. Rate limit — how fast?

Alice gets 5 requests per minute. Fire six:

In [ ]:
requests.post(f"{BASE}/_reset")   # reset the demo state
for i in range(1, 7):
    r = call("POST", "/v1/analyze", key="alice-123", body={"question": "hi", "tokens": 10})
    print(f"request {i}: {r.status_code}")


## The subtle part: a malformed request still burns a slot

Send five requests with a missing body (each 422), then one valid request:

In [ ]:
requests.post(f"{BASE}/_reset")
for i in range(1, 6):
    r = call("POST", "/v1/analyze", key="alice-123", body={})   # 422, but counted at the door
    print(f"bad body {i}: {r.status_code}")

r = call("POST", "/v1/analyze", key="alice-123", body={"question": "hi"})
print("now a valid one:", r.status_code, "<- the 422s already used the allowance")


## 3. Authorization — are you allowed?

Alice may read her own usage, not Bob's:

In [ ]:
requests.post(f"{BASE}/_reset")
r = call("GET", "/v1/usage/bob", key="alice-123")
print("alice -> bob's usage:", r.status_code, r.json()["detail"])

r = call("GET", "/v1/usage/alice", key="alice-123")
print("alice -> own usage:  ", r.status_code, r.json())


## 4. Quota — the monthly token allowance

Alice may spend 5000 tokens this month. Ask for 6000 in one call:

In [ ]:
requests.post(f"{BASE}/_reset")
r = call("POST", "/v1/analyze", key="alice-123", body={"question": "hi", "tokens": 6000})
print(r.status_code, r.json()["detail"])


## 5. Prepaid credits + top-up

Alice has 1000 credits (separate from the monthly allowance). Spend them, then top up:

In [ ]:
requests.post(f"{BASE}/_reset")

r = call("POST", "/v1/analyze", key="alice-123", body={"question": "h", "tokens": 800})
print("first 800:  ", r.status_code, "monthly_remaining:", r.json().get("monthly_remaining"))

r = call("POST", "/v1/analyze", key="alice-123", body={"question": "h", "tokens": 800})
print("second 800: ", r.status_code, r.json()["detail"])

r = call("POST", "/v1/credits/topup", key="alice-123", body={"amount": 5000})
print("top up:     ", r.status_code, r.json())

r = call("POST", "/v1/analyze", key="alice-123", body={"question": "h", "tokens": 800})
print("after top-up:", r.status_code, "monthly_remaining:", r.json().get("monthly_remaining"))


## Recap

> Three failures at three different doors: **401** you're not known, **429** you've asked too much, **422** your request is malformed — and a **402** means you've run out of paid tokens. The doors open in that order.

## What to try

* Change `REQUESTS_PER_MINUTE` in `app.py`, restart, and watch the 429 move.
* Send a valid key with a missing body: confirm it still consumes a rate slot.
* Top up Bob's credits, then spend them.
* Delete `recent` logic? No — instead, wait 60 seconds and confirm Alice's slots return.